<a href="https://colab.research.google.com/github/bvvnmanikanta/Scaler-AI-Notebooks/blob/main/class05_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 5 · RAG (Retrieval-Augmented Generation): class notebook

*AI & Agents: From Talking to AI to Building One · Scaler · Rishav Singh · v7, 14 Nov 2026*

**First: File → Save a copy in Drive.** Then this copy is yours to edit.

**How to use it.** When a slide says **Live demo**, **Run along** or **Hands-on** with *Notebook · Section N*, come to that section here.
- **To copy a prompt:** click inside its code cell, press **Ctrl+A** then **Ctrl+C** (Cmd on a Mac). Or run the cell and copy the printed text.
- Every code cell runs in Colab as it is. No API key needed today, except the optional Section 1c (it skips by itself without one).

**Today in one line:** give the AI the right notes before it answers. Retrieve the right pages → Augment (put them in front of the question) → Generate (answer from them, with a source).

## Section 0 · Tabs to open
1. **notebook.google.com**: Gemini Notebook, signed in with Google. Until July 2026 it was called **NotebookLM**; the old address still works.
2. **perplexity.ai**: works without logging in; logging in saves your sessions.
3. **chatgpt.com**
4. Later: **notegpt.io** (Section 5 and the mini build)

## Section 1 · Run along: RAG by hand *(slide 8 · 5 min)*

The cue card's code. It doesn't call any AI: it only builds and prints the prompt that **would** be sent, so you can see what RAG changes.

**Steps**
1. Run the next cell. Compare call 1 and call 2.
2. Run the cell after it, copy `CALL_2_PROMPT`, paste it into **ChatGPT**, send.
3. Post the refund rule ChatGPT gave you in the Zoom chat.

**Watch for:** call 1 and call 2 differ by one thing: the policy text above the question.

In [ ]:
def ask_llm(question, context=None):
    prompt = f"Context:\n{context}\n\nQuestion: {question}" if context else question
    print("---- PROMPT SENT TO THE LLM ----")
    print(prompt)

ask_llm("What's our company's refund policy?")

print()

ask_llm(
    "What's our company's refund policy?",
    context="Acme Corp refund policy (2026): full refund within 30 days of purchase, store credit after 30 days up to 90 days, no refunds after 90 days."
)

In [ ]:
# Copy this whole prompt into ChatGPT. You are the "retriever": you found the policy and pasted it in front.
CALL_2_PROMPT = """Context:
Acme Corp refund policy (2026): full refund within 30 days of purchase, store credit after 30 days up to 90 days, no refunds after 90 days.

Question: What's our company's refund policy?"""

print(CALL_2_PROMPT)

### Section 1b · Optional: a tiny retriever that finds the page by itself

Real RAG tools do steps 1 and 2 for you. This cell does a toy version in plain Python (no libraries, runs offline):
- the help centre is cut into **chunks** (one answer each),
- each chunk is **scored** against your question,
- the top 2 are pasted in front of the question.

A real system turns every chunk into an **embedding** (a list of numbers that captures meaning) and keeps them in a **vector database**. Here a small "same meaning" word list stands in for that, so "money back" can still find "refund".

Change `QUESTION` and run it again. Try: *How do I delete my account?* or *My phone broke, is it covered?*

In [ ]:
import re

HELP_CENTRE = {
    "Refunds": "Full refund within 30 days of purchase.",
    "Store credit": "After 30 days and up to 90 days, you get store credit.",
    "After 90 days": "No refunds or credit after 90 days.",
    "Shipping": "Orders ship in 2 working days. Free over Rs 999.",
    "Cancel an order": "Cancel any time before it ships, from My Orders.",
    "Warranty": "Electronics carry a 1-year warranty. Repairs are free.",
    "Payments": "UPI, cards and net banking. Cash on delivery up to Rs 5,000.",
    "Delete your account": "Settings > Delete account. Data is erased in 30 days.",
}

# ponytail: a hand-made "same meaning" list stands in for embeddings; use an embedding model + vector DB for real documents.
SAME_MEANING = {
    "money": "refund", "back": "refund", "refunds": "refund", "returned": "refund",
    "arrive": "ship", "arrives": "ship", "delivery": "ship", "deliver": "ship", "shipping": "ship", "ships": "ship",
    "orders": "order",
    "broke": "warranty", "broken": "warranty", "covered": "warranty", "repair": "warranty", "repairs": "warranty",
    "erase": "delete", "erased": "delete", "remove": "delete",
}
STOP = set("a an the i my me can will do does did get if in on at to of for be is are what when how have has and or with from it this that you your up by".split())

def words(text):
    """Lower-case words, minus filler words, with same-meaning words merged."""
    return {SAME_MEANING.get(w, w) for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}

def retrieve(question, k=2):
    """Step 1, Retrieve: score every chunk by shared words, keep the best k (ties keep help-centre order)."""
    q = words(question)
    scored = sorted(((len(q & words(t + " " + x)), t) for t, x in HELP_CENTRE.items()), key=lambda s: -s[0])
    return [t for score, t in scored[:k] if score > 0]

def augment(question, titles):
    """Step 2, Augment: put the retrieved chunks in front of the question."""
    context = "\n".join(f"[{i}] {t}: {HELP_CENTRE[t]}" for i, t in enumerate(titles, 1))
    return ("Answer only from the context below. Cite the chunk number like [1]. "
            "If the answer isn't there, say so.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}")

QUESTION = "Can I get my money back?"
top = retrieve(QUESTION)
print("Retrieved:", top, "\n")
print(augment(QUESTION, top))

# Self-check: meaning beats spelling. "money back" never appears in the help centre, yet Refunds comes first.
assert retrieve("Can I get my money back?")[0] == "Refunds"
assert retrieve("How many days until my order arrives?")[0] == "Shipping"
assert retrieve("How do I delete my account?")[0] == "Delete your account"

### Section 1c · Optional: real embeddings (needs a free Gemini API key)

Section 1b faked "same meaning" with a word list. Here Gemini's real embedding model, `gemini-embedding-2`, turns each phrase into a list of numbers (3,072 by default). **Cosine similarity** measures how close two lists point: near 1 = same meaning, lower = less related.

**Steps**
1. Get a free key at **aistudio.google.com** → Get API key. (You need it for Class 6 anyway.)
2. In Colab, click the **key icon** on the left (Secrets) → add `GEMINI_API_KEY` → turn on notebook access.
3. Run the cell. No key? It prints a note and skips. Nothing breaks.

**Watch for:** "money back" scores much closer to "refund" than to "delivery date", although it shares no words with either.

In [ ]:
# Section 1c: real embeddings. Skips cleanly if there's no GEMINI_API_KEY.
import os, math

def cosine(a, b):
    """How close two meanings are: near 1 = same direction, lower = less related."""
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

PHRASES = ["money back", "refund", "delivery date"]

key = os.environ.get("GEMINI_API_KEY")
if not key:
    try:
        from google.colab import userdata   # Colab Secrets (key icon on the left)
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        key = None

if not key:
    print("No GEMINI_API_KEY found, so this cell skips. Add one in Colab Secrets to see real embeddings.")
else:
    from google import genai               # preinstalled in Colab; elsewhere: pip install google-genai
    client = genai.Client(api_key=key)
    result = client.models.embed_content(model="gemini-embedding-2", contents=PHRASES)
    vectors = [e.values for e in result.embeddings]
    print(f"Each phrase became a list of {len(vectors[0])} numbers.")
    print(f'"money back" vs "refund":        {cosine(vectors[0], vectors[1]):.3f}')
    print(f'"money back" vs "delivery date": {cosine(vectors[0], vectors[2]):.3f}')

# Self-check of the maths (runs with or without a key)
assert abs(cosine([1, 0], [1, 0]) - 1) < 1e-9 and abs(cosine([1, 0], [0, 1])) < 1e-9

## Section 2 · Run along: build an LLM study notebook *(slide 12 · 13 min)*

**Gemini Notebook** (the cue card's NotebookLM) answers **only** from the sources you add, with citations to the exact passage. If the answer isn't in your sources, it says so.

**Steps**
1. notebook.google.com → **Create new notebook**.
2. **+ Add** → **Website** → paste `IBM_PAGE`. Then **+ Add** → **YouTube** → paste `KARPATHY_TALK`. Wait for each to finish (about 30 s).
3. Optional: download `STANFORD_PDF` and upload it as a third source.
4. Ask `Q1`. **Click a citation**: it jumps to the exact sentence (hover shows the quote).
5. Ask `Q2`, then `IPL_QUESTION`.
6. Studio panel → **Audio Overview** (it takes a few minutes; listen later).

**Post in Zoom chat:** ✅ when your first citation opens, then the message you got for the IPL question.

**Tip from the cue card:** start with "According to my sources…". It keeps the answer anchored, like a scope prefix on a query.

YouTube sources need a **public video with captions**. If one fails, skip it.

In [ ]:
IBM_PAGE = "https://www.ibm.com/think/topics/large-language-models"
KARPATHY_TALK = "https://youtu.be/zjkBMFhNj_g"          # Andrej Karpathy, "[1hr Talk] Intro to Large Language Models"
STANFORD_PDF = "https://drive.google.com/file/d/14ZK_FQ68yeCvEhpB-0yS0nhmomP_fIKV/view"   # "What are LLMs - Stanford.pdf"

Q1 = "According to my sources, how is a large language model trained?"
Q2 = "According to my sources, what is a jailbreak? Which source explains it?"
IPL_QUESTION = "Who won the IPL final in 2026?"

print(IBM_PAGE, KARPATHY_TALK, STANFORD_PDF, "", Q1, Q2, IPL_QUESTION, sep="\n")

## Section 3 · Question shapes: what each one retrieves *(slide 14 · 2 min)*

Class 3 taught templates with placeholders. The new part: **the shape of your question changes which passages the retriever pulls in**, before the model writes a word.

| Shape | Retrieval pulls in |
|---|---|
| Factual | the one best passage |
| Comparison | one passage from each source you name |
| Synthesis | many passages across all your sources |
| Explain like I am five | a few passages, rewritten simply |
| Devil's advocate | passages that disagree, or "none" |

**Steps**
1. Pick one shape below. Replace the `[brackets]` (ideas: *context window*, *tokens*, *jailbreak*, *prompt injection*, *fine-tuning*; for Comparison: *the IBM page* and *Karpathy's talk*).
2. Run it in your Gemini Notebook.
3. **Count the citation numbers** in the answer. Post the shape and the count in Zoom chat, like `Synthesis 6`.

These are the cue card's five templates, word for word.

In [ ]:
import re

TEMPLATES = {
    "FACTUAL": "According to my sources, what is [concept]?",

    "COMPARISON": "Where do [source 1] and [source 2] disagree about [topic]?",

    "SYNTHESIS": """Based on all my sources, what are the three most important
things a beginner should know about [topic]? Cite which
source each point comes from.""",

    "EXPLAIN LIKE I AM FIVE": """Using only my uploaded sources, explain [complex concept]
as if I've never heard of it before.""",

    "DEVIL'S ADVOCATE": """What are the strongest objections to the main argument
in my sources? Cite which source raises each concern
(or note if none do).""",
}

def fill(template, **values):
    """Swap each [placeholder] for your own words: fill(t, concept="context window")."""
    for key, value in values.items():
        template = re.sub(r"\[" + key.replace("_", " ") + r"\]", value, template)
    return template

print(fill(TEMPLATES["FACTUAL"], concept="a context window"))
print(fill(TEMPLATES["COMPARISON"], source_1="the IBM page", source_2="Karpathy's talk", topic="how LLMs are trained"))

**My citation tally** *(double-click to type)*

| Shape I ran | Citations in the answer | From how many sources? |
|---|---|---|
| | | |
| | | |

When the class posts its numbers, compare: factual answers usually carry 1 or 2 citations, synthesis answers 5 or more from more than one source. Same notebook, same sources: the shape changed what got retrieved.

## Section 4 · Live demo and run along: Perplexity *(slides 17–18)*

Perplexity is the same RAG pattern, but step 1 (Retrieve) searches the **live web**. Every answer has numbered citations you can click.

**Slide 17 · same question, two tools (watch).** ChatGPT gets `MEMORY_ONLY` + `NEWS_QUESTION`; Perplexity gets `NEWS_QUESTION` alone. ChatGPT has **no web-search off switch** on personal plans; it searches when it thinks it should. So we ask it not to, and check the reply has no **Sources** button. If it searched anyway, that's RAG too.

**Slide 18 · your turn.**
1. perplexity.ai → paste `SEARCH_QUESTION`.
2. Hover a citation number to preview it, then click it.
3. **Post ✅ or ❌:** does the page really say what the answer says?

**Watch only: Deep Research.** In the question box, open the **+** menu → **Deep Research** ("Included in Pro"; free accounts get very few runs, about one a month). It runs dozens of searches and writes a cited report, usually in 2–4 minutes. **Watch for:** count the searches in its progress list. Search, read, spot the gap, search again: retrieval in a loop, which Class 7 calls an agent. The cue card's "Focus" source picker is now an Enterprise-only feature called *Choose sources*: choosing Web, Academic or your company's files just changes where step 1 looks.

In [ ]:
MEMORY_ONLY = "Do not search the web. Answer only from what you already know, and say if you are unsure.\n\n"

NEWS_QUESTION = "What did Anthropic announce in the last 30 days?"
NEWS_QUESTION_BACKUP = "What did OpenAI announce in the last 30 days?"   # if Anthropic has been quiet

SEARCH_QUESTION = "What are the top 3 open-source LLMs released in 2026 so far?"

RESEARCH_PROMPT = "Compare the architectural trade-offs between RAG and fine-tuning for enterprise AI deployments in 2026."

print("ChatGPT gets:\n" + MEMORY_ONLY + NEWS_QUESTION)
print("\nPerplexity gets:\n" + NEWS_QUESTION)
print("\nYour search:\n" + SEARCH_QUESTION)

## Section 5 · Live demo: NoteGPT, one long video *(slide 20 · watch)*

NoteGPT transcribes a YouTube talk, cuts the transcript into chunks by time, and answers questions with a **timestamp** you can click.

**Steps**
1. notegpt.io → **YouTube** tab → paste `TALK_URL` → **Generate Summary** (about 30 s; it may ask you to sign in, and free accounts get a small monthly quota).
2. Read the timestamped summary.
3. Ask `TIMESTAMP_QUESTION` in NoteGPT's chat. Click the timestamp in the answer.

**Guess first:** at what minute does he first talk about microservices? Type a number in chat.

**The chaining pattern** (the cue card): copy NoteGPT's notes → paste them into your Gemini Notebook as a **pasted-text** source → now the video is searchable next to your PDFs and web pages. Pre-process with tool 1, ground with tool 2: that's how production RAG pipelines handle meeting recordings.

**If NoteGPT is down:** add the same YouTube link to your Gemini Notebook (it reads public YouTube transcripts directly), or use `BACKUP_TALK_URL`, whose chapters are known.

In [ ]:
TALK_URL = "https://youtu.be/oejXFgvAwTA"   # Randy Shoup, "Large-Scale Architecture: The Unreasonable Effectiveness of Simplicity" (YOW! 2022)
TIMESTAMP_QUESTION = "At what timestamp does the speaker first talk about microservices?"

BACKUP_TALK_URL = "https://youtu.be/zjkBMFhNj_g"   # Karpathy, chapters from the video description:
KARPATHY_CHAPTERS = {"LLM OS": "42:15", "LLM security intro": "45:43", "Jailbreaks": "46:14", "Prompt injection": "51:30", "Data poisoning": "56:23"}

print(TALK_URL, TIMESTAMP_QUESTION, "", "Backup:", BACKUP_TALK_URL, KARPATHY_CHAPTERS, sep="\n")

## Section 6 · Hands-on: Exercises 5.1 and 5.2 *(slides 24–25)*

Three tabs side by side: **ChatGPT** (put `MEMORY_ONLY` in front), **Perplexity**, **your Gemini Notebook** from Section 2.

### Exercise 5.1 · one question, three tools (8 min)
Pick **one** prompt below and ask it, word for word, in all three tabs.
- **A** tests freshness. *Next.js* = a popular framework for building websites with React.
- **B** tests niche knowledge. *Kubernetes* runs your app on many servers; a *pod* = one running copy of your app.
- **C** only your notebook can answer.

Ask yourself: Which answer **sounded** most confident? Was that confidence justified? Which one showed **where** the information came from? Which one was willing to say **"I don't know"**?

**Post in Zoom chat:** `surest = C, P or N` and `showed sources = C, P or N`.

### Exercise 5.2 · the trick question (4 min)
Ask `TRICK_QUESTION` in all three tabs. This is the **false premise test**: AI teams ask something wrong on purpose and check whether the AI pushes back or plays along.

**Post in Zoom chat:** the tool that corrected you.

Don't worry if you didn't know the answer to the trick question. The point is how the tools handle a wrong assumption, not your Python knowledge.

In [ ]:
PROMPT_A = "What are the three biggest changes in Next.js in the last 6 months?"

PROMPT_B = """What are the most common production failure modes of
Kubernetes pods and their typical root causes?"""

PROMPT_C = """Based on my uploaded sources, what are the three key takeaways
from the material I've studied?"""

TRICK_QUESTION = "Why does Python require you to compile your code before running it?"

print("ChatGPT version of A:\n" + MEMORY_ONLY + PROMPT_A)

| My check | ChatGPT (memory only) | Perplexity | My notebook |
|---|---|---|---|
| Sounded sure? | | | |
| Showed sources? | | | |
| Said "I don't know"? | | | |
| Trick question: corrected me or played along? | | | |

*Double-click this cell to type in it.*

## Section 7 · Hands-on: mini build, pick one *(slide 29 · 12 min, finish at home)*

### A · Priya's Placement Prep (Gemini Notebook) · best pick for most of you
Priya is prepping for placements: lecture notes, a system design book, interview-prep blog bookmarks. She wants one assistant that answers "based on my notes" without making things up.
1. New notebook: **Priya's Placement Prep**.
2. At least **4 sources, mixed**: 1 PDF (a textbook chapter or cheat sheet) · 1 YouTube talk (system design or DSA) · 1 web page (GeeksforGeeks, InterviewBit…) · 1 pasted text (your own notes).
3. Ask the three questions in `PRIYA_QUESTIONS`.
4. Studio: **Study guide** (under Reports) + **Audio Overview**.

**Done when** every answer has clickable citations to real passages. No citations? Check that your sources finished uploading.

### B · Arjun's AI News Brief (Perplexity)
Arjun writes a weekly AI newsletter. Every Friday he needs what actually happened in AI in the last 7 days.
1. perplexity.ai, logged in (saves the session).
2. **+** menu → **Deep Research** (free accounts get about one run a month; normal Search with the same prompt still works).
3. Run `ARJUN_PROMPT`.
4. **Verify 3 citations**: click them. Do the pages say what the brief says?

**Done when** every claim has a citation and 3 of them open real articles from the last 7 days. Save it in a Perplexity **Project** and re-run it every Friday. RAG cuts made-up facts a lot; it doesn't end them, so always spot-check.

### C · Kavya's Lecture Library (NoteGPT → Gemini Notebook) · best as homework
Kavya is learning backend engineering from long conference talks.
1. Pick **3 long talks on one topic** (search "GOTO Conferences", "InfoQ" or "Strange Loop"; or three long DSA, DBMS or Python lectures).
2. For each: notegpt.io → paste the link → generate notes → copy them.
3. New notebook **Kavya's Lecture Library**; paste each talk's notes as a separate **pasted-text** source, with the video link in the title.
4. Ask the questions in `KAVYA_QUESTIONS`.

**Done when** one answer cites two or more talks.

In [ ]:
PRIYA_QUESTIONS = [
    "Based on my sources, what are the key differences between SQL and NoSQL databases?",
    "Across all my uploaded sources, what's the most commonly mentioned system design pattern?",
    "What would be my weakest topic if an interviewer asked from my uploaded materials? Cite examples.",
]

ARJUN_PROMPT = """Produce a weekly AI news brief covering only the last 7 days.
Include: new model releases, significant funding rounds,
important research papers, and major product launches.
Cite sources for every claim. Flag anything where sources
disagree. End with one sentence on what mattered most and why."""

KAVYA_QUESTIONS = [
    "Which talk best explains [concept]?",
    "Where do the three speakers disagree?",
    "What's the one piece of advice all three speakers emphasise?",
]

print(*PRIYA_QUESTIONS, sep="\n")

## Section 8 · Homework and links

**Homework**
1. Finish your mini build (A, B or C). Then try a second one.
2. Class 5 assignment: [RAG_Assessment.ipynb](https://drive.google.com/file/d/1qTwxK3H_ZnXPnwm7EPY3d2hfSZwjV-Oo/view?usp=sharing) (also on your Scaler dashboard).
3. Post-read: [Class5_RAG_postRead.ipynb](https://drive.google.com/open?id=1JvTKXlD5bEz7v_CFLLOPNUcaELUCS2Dr&usp=drive_copy).
4. Stretch: try hybrid search (the last cell of this notebook).
5. Pre-read (if you missed it): [Class 5: RAG, Retrieval-Augmented Generation · Pre-Read](https://drive.google.com/open?id=129SS6ycf-5I2SdHzSIFDI9Z01ixWmO4ohc7muZUXayE&usp=drive_copy).

**Before Class 6 (n8n automation)**
- An **n8n** account (free cloud trial at n8n.io).
- A **Gemini API key** from aistudio.google.com (free tier).
- Optional: a **Telegram** bot from @BotFather.

**Today's tools**
- notebook.google.com (Gemini Notebook, was NotebookLM) · free, 50 sources per notebook without a paid plan
- perplexity.ai · free: a few Pro Searches a day, Deep Research about once a month
- notegpt.io · small free monthly quota
- chatgpt.com

**One line to remember:** a grounded AI that says "I don't know" is worth more than a clever one that lies.

### Stretch · try hybrid search *(homework, 15 min)*

Real systems don't pick exact words **or** meaning: they run **both** searches and merge the two ranked lists (**hybrid search**), usually with **Reciprocal Rank Fusion (RRF)**: each list gives every result `1 / (60 + its rank)`, and the scores add up. Then a **reranker** (a second model) re-reads the shortlist and keeps the best few. Azure AI Search and OpenAI's file search both work this way.

The cell builds on Section 1b (run Section 1b first). Then:
1. Run it and compare the three lists for each question.
2. Add a chunk to `HELP_CENTRE`, like `"Card error PAY-402": "PAY-402 means your bank declined the card. Try UPI."`, and ask `"What is PAY-402?"`.
3. In this toy both searches use words, so they often agree. Real embeddings often rank rare codes like `PAY-402` poorly; keyword search catches them. That's why teams run both.

In [ ]:
# Hybrid search on the Section 1b help centre: two rankings, one merged list (Reciprocal Rank Fusion).
import re

if "HELP_CENTRE" not in globals():
    raise SystemExit("Run Section 1b first: it defines HELP_CENTRE, STOP and words().")

def keyword_rank(question):
    """Exact words only, like Ctrl+F: no 'same meaning' list."""
    q = {w for w in re.findall(r"[a-z0-9]+", question.lower()) if w not in STOP}
    scores = {t: len(q & set(re.findall(r"[a-z0-9]+", (t + " " + x).lower()))) for t, x in HELP_CENTRE.items()}
    return [t for t, s in sorted(scores.items(), key=lambda kv: -kv[1]) if s > 0]

def meaning_rank(question):
    """Section 1b's search: same-meaning words merged (our stand-in for embeddings)."""
    q = words(question)
    scores = {t: len(q & words(t + " " + x)) for t, x in HELP_CENTRE.items()}
    return [t for t, s in sorted(scores.items(), key=lambda kv: -kv[1]) if s > 0]

def rrf(*rankings, k=60):
    """Reciprocal Rank Fusion: each ranking gives 1 / (k + rank) to each result; add them up."""
    total = {}
    for ranking in rankings:
        for rank, title in enumerate(ranking, 1):
            total[title] = total.get(title, 0) + 1 / (k + rank)
    return sorted(total, key=total.get, reverse=True)

for q in ["Can I get my money back?", "How do I delete my account?"]:
    print(q)
    print("  keyword:", keyword_rank(q)[:3])
    print("  meaning:", meaning_rank(q)[:3])
    print("  hybrid: ", rrf(keyword_rank(q), meaning_rank(q))[:3])

# Self-check: exact words miss "money back", the merged list still puts Refunds first.
assert keyword_rank("Can I get my money back?") == []
assert rrf(keyword_rank("Can I get my money back?"), meaning_rank("Can I get my money back?"))[0] == "Refunds"
assert rrf(keyword_rank("How do I delete my account?"), meaning_rank("How do I delete my account?"))[0] == "Delete your account"